In [ ]:
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def find_file(filename):
    matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
    if not matches:
        raise FileNotFoundError(f"Nu am gasit {filename}. Verifica panoul Input din dreapta!")
    return matches[0]

train = pd.read_csv(find_file('train_data.csv'))
test = pd.read_csv(find_file('test_data.csv'))
sample = pd.read_csv(find_file('sample_output.csv'))

print(f"Train incarcat: {train.shape}, Test incarcat: {test.shape}")

EVASION_CLASSES = [
    'Explicit',
    'Implicit',
    'General',
    'Partial/half-answer',
    'Dodging',
    'Deflection',
    'Declining to answer',
    'Claims ignorance',
    'Clarification'
]

EVASION_TO_CLARITY = {
    'Explicit': 'Clear Reply',
    'Implicit': 'Ambivalent',
    'General': 'Ambivalent',
    'Partial/half-answer': 'Ambivalent',
    'Dodging': 'Ambivalent',
    'Deflection': 'Ambivalent',
    'Declining to answer': 'Clear Non-Reply',
    'Claims ignorance': 'Clear Non-Reply',
    'Clarification': 'Clear Non-Reply'
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train, train['evasion_label'])):
    train.loc[val_idx, 'fold'] = fold

# Salvam folds.csv si ordinea claselor pentru echipa
train[['datapointID', 'fold', 'clarity_label', 'evasion_label']].to_csv('folds.csv', index=False)
with open('classes_order.txt', 'w') as f:
    for idx, cls in enumerate(EVASION_CLASSES):
        f.write(f"{idx}: {cls}\n")

train['is_test'] = 0
test['is_test'] = 1
full = pd.concat([train, test], ignore_index=True)
full = full.sort_values(['url', 'question_order']).reset_index(drop=True)

same_url = full['url'] == full['url'].shift(1)
full['turn_gap'] = np.where(same_url, full['question_order'] - full['question_order'].shift(1), -1)
full['prev_question'] = np.where(full['turn_gap'] == 1, full['interview_question'].shift(1), "")
full['prev_answer'] = np.where(full['turn_gap'] == 1, full['interview_answer'].shift(1), "")

context_cols = full[['datapointID', 'turn_gap', 'prev_question', 'prev_answer']]
train_ctx = train.drop(columns=['is_test']).merge(context_cols, on='datapointID', how='left')
test_ctx = test.drop(columns=['is_test']).merge(context_cols, on='datapointID', how='left')

train_ctx.to_csv('train_context.csv', index=False)
test_ctx.to_csv('test_context.csv', index=False)
print("Am salvat folds.csv, classes_order.txt, train_context.csv si test_context.csv!")

label2id = {c: i for i, c in enumerate(EVASION_CLASSES)}
y_train = train_ctx['evasion_label'].map(label2id).values

def build_text(df):
    return (
        "Presedinte: " + df['president'].fillna('') +
        " | Context: " + df['context'].fillna('') +
        " | Intrebare: " + df['interview_question'].fillna('') +
        " | Raspuns: " + df['interview_answer'].fillna('')
    )

X_train_text = build_text(train_ctx)
X_test_text = build_text(test_ctx)

oof_probs = np.zeros((len(train_ctx), len(EVASION_CLASSES)))
test_probs = np.zeros((len(test_ctx), len(EVASION_CLASSES)))

for fold in range(5):
    tr_mask = train_ctx['fold'] != fold
    va_mask = train_ctx['fold'] == fold
    
    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True)
    X_tr = vec.fit_transform(X_train_text[tr_mask])
    X_va = vec.transform(X_train_text[va_mask])
    X_te = vec.transform(X_test_text)
    
    clf = LogisticRegression(C=2.0, max_iter=1000, class_weight='balanced')
    clf.fit(X_tr, y_train[tr_mask])
    
    oof_probs[va_mask] = clf.predict_proba(X_va)
    test_probs += clf.predict_proba(X_te) / 5.0

oof_pred_9 = [EVASION_CLASSES[i] for i in oof_probs.argmax(axis=1)]
oof_pred_3 = [EVASION_TO_CLARITY[c] for c in oof_pred_9]

f1_sub1 = f1_score(train_ctx['clarity_label'], oof_pred_3, average='macro')
f1_sub2 = f1_score(train_ctx['evasion_label'], oof_pred_9, average='macro')
print(f"\n>>> Scor CV Baseline - Subtask 1 (Clarity Macro F1): {f1_sub1:.4f}")
print(f">>> Scor CV Baseline - Subtask 2 (Evasion Macro F1): {f1_sub2:.4f}")

clarity_classes = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
test_clarity_probs = np.zeros((len(test_ctx), 3))
for j, c_name in enumerate(EVASION_CLASSES):
    parent_idx = clarity_classes.index(EVASION_TO_CLARITY[c_name])
    test_clarity_probs[:, parent_idx] += test_probs[:, j]

pred_3 = [clarity_classes[i] for i in test_clarity_probs.argmax(axis=1)]
pred_9 = [EVASION_CLASSES[i] for i in test_probs.argmax(axis=1)]

map_sub1 = dict(zip(test_ctx['datapointID'], pred_3))
map_sub2 = dict(zip(test_ctx['datapointID'], pred_9))

sub = sample[['subtaskID', 'datapointID']].copy()
sub['answer'] = np.where(
    sub['subtaskID'] == 1,
    sub['datapointID'].map(map_sub1),
    sub['datapointID'].map(map_sub2)
)

assert len(sub) == 592, "Eroare: trebuie sa fie exact 592 randuri!"
assert sub['answer'].isnull().sum() == 0, "Eroare: exista valori lipsa!"

sub.to_csv('submission_baseline.csv', index=False)
print("\nSucces! Fisierul submission_baseline.csv a fost generat si verificat.")

Train incarcat: (3278, 11), Test incarcat: (296, 9)
Am salvat folds.csv, classes_order.txt, train_context.csv si test_context.csv!
